### 这篇notebook是使用稀疏表示(SR)方法来对电流瞬态数据进行处理的示例

In [1]:
%matplotlib widget
import sys
sys.path.append("..")
import OpenDLTS_DataHandler as oddh
import numpy as np
import matplotlib.pyplot as plt
import cvxpy as cp
from pathlib import Path

导入数据

In [2]:
data = oddh.Data_Loader(
    transient_data = './data/IDLTS_data.transdata',
    raw_data_scaling_factor = 1.0,#导入数据时对数据进行缩放，如果数据不是标准单位比如 [F/A/V]
    data_scaling_factor = 1e6,#根据数据的量级进行缩放，缩放后的数据会直接被应用到后续拟合过程中，缩放后太小/太大的数据可能会导致不收敛。一般对于pF量级的电容数据是1e12，对于μA量级的电流数据是1e6
    data_type = 'I',#确定数据的类型，电容数据是'C'，电流数据是'I'，电压数据是'V'，这会影响图片中单位的显示，不影响数据拟合
    #data_x_type = 'Time',#确定数据的x类型，一般是瞬态数据，因此默认即可
    #condition_type = 'Temperature',#确定数据的测量情况，一般在不同温度下进行测量，因此默认即可
    logging_level = 'info',
    logging_file = None,#如果需要将log写入某一文件，指定路径字符串
    logging_file_clear = False,#导入数据时清除logging_file路径下的log
)

(ODDH) 2026-04-01 10:56:17,856 - __oddh__ - INFO - Data loaded from file: /mnt/userdata/yutao/jupyter/LDLTS-Tao/OpenDLTS_DataHandler/examples/data/IDLTS_data.transdata


### 数据预处理

In [3]:
# 实际的Ns可能不等于设定的num，详细的算法参考文件_ReSampleFromTimeArray.py
ldlts_data = data.data_space(num=100,space='log')
ui = oddh.Widgets.LDLTS_Viewer_Box(
    dlts_data = ldlts_data,
    Ns = 500,
    s0 = 1e-1,
    s1 = 1e5
)

(ODDH) 2026-04-01 10:56:18,960 - __oddh__ - INFO - Data loaded from provided dictionary.
(ODDH) 2026-04-01 10:56:18,960 - __oddh__ - INFO - New Data resampled to 100 points in log space.
(ODDH) 2026-04-01 10:56:18,961 - __oddh__.LDLTS_Viewer_Box - INFO - LDLTS Method Initialized with Ns=500, s0=1.0E-01, s1=1.0E+05


### 使用Ti_list选择部件

In [4]:
_Ti_list_widget=oddh.Widgets.Ti_list_selector(ui.SR.T,Ti0=0,Ti1=None,min_step_coeff=1.68,init_showui=False)
class temp_Ti_class:
    def __init__(self):
        pass
    @property
    def value(self):
        return _Ti_list_widget.selected_Ti
Ti=temp_Ti_class()

In [5]:
_Ti_list_widget.show_ui()
def Ti_list():
    return _Ti_list_widget.Ti_list_selected

### 设置一个比较小的正则化参数，使用L1方法进行过拟合

In [6]:
polarity='negative'# 约束数据由负的TCS构成
material='sic'# 指定计算活化能与捕获截面使用的材料参数
# 指定计算活化能与捕获截面使用的材料的多子类型
# 对于P型。如果TCS为正，则认定为多子（空穴）陷阱，使用空穴相关参数计算其捕获截面；如果TCS为负，则认定为少子（电子）陷阱，使用电子相关参数计算其捕获截面；
# 对于N型。如果TCS为正，则认定为多子（电子）陷阱，使用电子相关参数计算其捕获截面；如果TCS为负，则认定为少子（空穴）陷阱，使用空穴相关参数计算其捕获截面。
material_doping_type='P'
solve_arg = {}
solve_arg['L1'] = {
    "Ti_list":np.arange(len(ui.L1.T)),# 指定温度的index list，这些温度下的数据会被拟合，这里使用所有温度
    "polarity":polarity,
    "lambda1":1e-8,# 指定正则化参数
    "verbose":True,
    "skip_solved":True,
    #"solver":'MOSEK',
    "solver":'GUROBI',# 指定求解器，GUROBI/MOSEK/COPT需要申请和配置许可证，学生可以免费申请获得
    #"solver":'COPT',
    #"solver_params":{'eps':1e-9,'canon_backend':cp.SCIPY_CANON_BACKEND},
    "solver_params":{'Threads':0,'Presolve':-1,'BarOrder':0,'Method':2,'canon_backend':cp.SCIPY_CANON_BACKEND},
    'f_scaling_by_C':False,
    'enable_irls_mode':False,
    'irls_weight':1,
    'arg_lists_use_cp_parameter':['lambda1']
}
ui.L1.solve(**solve_arg['L1'])

(ODDH) 2026-04-01 10:56:22,414 - __oddh__.L1 - INFO - L1 solve: build new problem
(ODDH) 2026-04-01 10:56:22,418 - __oddh__.L1 - INFO - #CVXPY Output#: ===============================================================================
(ODDH) 2026-04-01 10:56:22,418 - __oddh__.L1 - INFO - #CVXPY Output#:                                      CVXPY                                     
(ODDH) 2026-04-01 10:56:22,418 - __oddh__.L1 - INFO - #CVXPY Output#:                                      v1.7.2                                    
(ODDH) 2026-04-01 10:56:22,419 - __oddh__.L1 - INFO - #CVXPY Output#: ===============================================================================
(CVXPY) 2026-04-01 10:56:22,419 - __cvxpy__ - INFO - Your problem has 62625 variables, 62500 constraints, and 1 parameters.
(CVXPY) 2026-04-01 10:56:22,419 - __cvxpy__ - INFO - It is compliant with the following grammars: DCP, DQCP
(CVXPY) 2026-04-01 10:56:22,419 - __cvxpy__ - INFO - CVXPY will first compile your pro

### 使用PRSSE分析过拟合时对应发射率的上下限

In [7]:
if False:
    prsse_arg = {
        'solve_index':-1,
        'si_list':None,
        "solver":'MOSEK',
        #"solver":'GUROBI',
        "solver_params":{'eps':1e-9,'canon_backend':cp.SCIPY_CANON_BACKEND},
        #"solver_params":{'Threads':0,'Presolve':-1,'BarOrder':0,'Method':2,'canon_backend':cp.SCIPY_CANON_BACKEND},
        "fwhm_height":0.5,
        "perturbation_factor":1
    }
    ui.L1.prsse(**prsse_arg)
    print(ui.L1.prsse_solve_history[0]['fwhm_s0_s1_array'][0])
    lower_em,upper_em = tuple(ui.L1.prsse_solve_history[0]['fwhm_s0_s1_array'][0])
else:
    lower_em,upper_em = tuple([25.10073217, 10998.73250164])

### 定义字典中的词条
方法get_dictionary_word_cover_arrh_sparse的原理图：
![Arrh_Plot_Fitting_Result](pic/arrh_gen_example.png)

In [8]:
wf_list = []
en_arg_list = []
unen_arg_list = []
w_polarity_list = []
w_dc_mono_list = []
w_constraint_prefactor_list = []

# Trap arrh
wf = oddh.LDLTS_Method.Word_Function.get_dictionary_word_cover_arrh_sparse
Ea_max = 1.500    # E_{a1} in pic
Ea_min = 0.200    # E_{a0} in pic
Ea_num = 200
limited_Ea_range = np.tan(np.linspace(np.arctan(Ea_min),np.arctan(Ea_max),Ea_num))
ref_T0 = np.min(ui.SR.T[Ti_list()])    # T_0 in pic
ref_T1 = np.max(ui.SR.T[Ti_list()])    # T_1 in pic
T_num = 200    # number of points need to be enumerated in pic

unen_arg = {
    'Ti_list':Ti_list(),
    'upper_em':upper_em,    # τ_0 in pic
    'lower_em':lower_em,    # τ_1 in pic
    'T_num':T_num,
    'upper_T':ref_T1,
    'lower_T':ref_T0,
    'T_power':2,
    'manual_T_list':None,
    'B_sp_n':6,
    'B_sp_degree':2
}
en_arg = {
    'Ea':limited_Ea_range,
    'word_index':np.arange(T_num),
    'B_sp_basis_i':np.arange(unen_arg['B_sp_n'])
}
# Additional polarity and monotonicity constraints
w_polarity = 'nonpos'
w_dc_mono = 'decrease'
wf_list.append(wf)
en_arg_list.append(en_arg)
unen_arg_list.append(unen_arg)
w_polarity_list.append(w_polarity)
w_dc_mono_list.append(w_dc_mono)
# Penalty weights of sub-dictionaries
w_constraint_prefactor_list.append(1)



# Trap Constant time constant
wf = oddh.LDLTS_Method.Word_Function.get_dictionary_word_constant_emission_sparse
unen_arg = {
    'Ti_list':Ti_list(),
    'manual_T_list':None,
    'B_sp_n':6,
    'B_sp_degree':2
}
en_arg = {
    'constant_emission_rate':np.logspace(np.log10(1e1),np.log10(1e5),400),
    'B_sp_basis_i':np.arange(unen_arg['B_sp_n'])
}
w_polarity = 'nonpos'
w_dc_mono = 'decrease'
wf_list.append(wf)
en_arg_list.append(en_arg)
unen_arg_list.append(unen_arg)
w_polarity_list.append(w_polarity)
w_dc_mono_list.append(w_dc_mono)
w_constraint_prefactor_list.append(0)

In [9]:
solve_arg['SR']={
    'Ti_list':Ti_list(),
    'verbose':True,
    'polarity':polarity,
    'lambda1':5e-2,
    'word_fun_list':wf_list,
    'enumerate_args_list_dic_list':en_arg_list,
    'unenumerate_args_list_dic_list':unen_arg_list,
    'polarity_list':w_polarity_list,
    'dc_mono_list':w_dc_mono_list,
    'constraint_prefactor_list':w_constraint_prefactor_list,
    #'constraint_max_rms':True,
    'constraint_max_rms':False,
    'skip_solved':True,
    #'solver':'MOSEK',
    #'solver_params':{'eps':1e-9,'canon_backend':cp.SCIPY_CANON_BACKEND}
    
    'solver':'GUROBI',
    'solver_params':{'Threads':0,'Presolve':-1,'BarOrder':-1,'Method':-1,'canon_backend':cp.SCIPY_CANON_BACKEND}
    
    #'solver':'CUOPT',
    #'solver_params':{'CUOPT_METHOD':'PDLP'}
    
    #'solver':'COPT',
    #'solver_params':{'LpMethod':6,'Crossover':0}
    
    #'enable_irls_mode':True,
    #'arg_lists_use_cp_parameter':['irls_weight'],
    #'irls_weight':np.abs(ldlts_method.solve_history[-1]['irls_target'])
    #'arg_lists_use_cp_parameter':['lambda1']
    #'solver_params':{}
}

import threading
def solve_wrapper():
    try:
        result = ui.SR.solve(**solve_arg['SR'])
    except Exception as e:
        print(f"{e}")

solve_thread = threading.Thread(target=solve_wrapper)
solve_thread.start()

(ODDH) 2026-04-01 10:56:45,568 - __oddh__.SR - INFO - SR solve: build new problem
(ODDH) 2026-04-01 10:56:45,573 - __oddh__.SR - INFO - Start enumerating all atoms...
(ODDH) 2026-04-01 10:56:45,573 - __oddh__.SR - INFO - Creating the input parameter list...
(ODDH) 2026-04-01 10:56:45,775 - __oddh__.SR - INFO - Creating the input parameter list success. Wall time: 0.20s
(ODDH) 2026-04-01 10:56:45,775 - __oddh__.SR - INFO - Start multi-process running
(ODDH) 2026-04-01 10:56:45,969 - __oddh__.SR - INFO - Further creating parameter lists. Wall time: 0.19s
(ODDH) 2026-04-01 10:56:47,753 - __oddh__.SR - INFO - multi-process running success. Wall time: 1.98s
(ODDH) 2026-04-01 10:56:47,754 - __oddh__.SR - INFO - Start assembling Dictionary
(ODDH) 2026-04-01 10:56:47,898 - __oddh__.SR - INFO - Assembling dictionary success. Wall time: 0.14s
(ODDH) 2026-04-01 10:56:47,898 - __oddh__.SR - INFO - Total length of non-zero value=7745700. Total wall time: 2.33s
(ODDH) 2026-04-01 10:56:49,734 - __odd